In [ ]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="st_all_minilm_l6_l2_distance_threshold_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [1]:
import torch
import numpy as np
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

device: mps


In [2]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(model_name, device=str(device))
print(model_name)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


sentence-transformers/all-MiniLM-L6-v2


In [3]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647


In [4]:
batch_size = 128

emb1 = model.encode(
    sent1,
    batch_size=batch_size,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

emb2 = model.encode(
    sent2,
    batch_size=batch_size,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print("emb1 shape:", tuple(emb1.shape))
print("emb2 shape:", tuple(emb2.shape))
print("embedding device:", emb1.device)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

emb1 shape: (408, 384)
emb2 shape: (408, 384)
embedding device: mps:0


In [ ]:
emb1_list = emb1.cpu().float().tolist()
emb2_list = emb2.cpu().float().tolist()
vault.create_embedding_list("sentence-transformers-sentence-1-distance_threshold", ndim=384)
vault.create_embedding_list("sentence-transformers-sentence-2-distance_threshold", ndim=384)

for i in range(len(emb1_list)):
    vault.append_embedding("sentence-transformers-sentence-1-distance_threshold", emb1_list[i], 
                       input_items = {"glue_mrpc_validation": [i, i + 1]}
                       )
    vault.append_embedding("sentence-transformers-sentence-2-distance_threshold", emb2_list[i], 
                       input_items = {"glue_mrpc_validation": [i, i + 1]}
                       )


description = "INSERT TEXT HERE ABOUT sentence-transformers-sentence-1-distance_threshold"
embedding = get_embeddings(description)
vault.create_description("sentence-transformers-sentence-1-distance_threshold", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("sentence-transformers-sentence-1-distance_threshold", cat, embedding, prop)

description = "INSERT TEXT HERE ABOUT sentence-transformers-sentence-2-distance_threshold"
embedding = get_embeddings(description)
vault.create_description("sentence-transformers-sentence-2-distance_threshold", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("sentence-transformers-sentence-2-distance_threshold", cat, embedding, prop)

In [5]:
distance_threshold = 0.63

distances = torch.linalg.vector_norm(emb1 - emb2, ord=2, dim=1)
y_pred_tensor = (distances <= distance_threshold).to(torch.int64)

distances_np = distances.detach().cpu().numpy()
y_pred = y_pred_tensor.detach().cpu().numpy()

print("done")
print("distance_threshold:", distance_threshold)
print("distance stats:", {
    "min": float(distances_np.min()),
    "max": float(distances_np.max()),
    "mean": float(distances_np.mean()),
})

done
distance_threshold: 0.63
distance stats: {'min': 0.07653803378343582, 'max': 1.1821868419647217, 'mean': 0.593309223651886}


In [ ]:
vault.create_record_list("sentence_transformers_mrpc_prediction_distance", column_names=["prediction"])

for i in range(len(y_pred)):
    vault.append_record("sentence_transformers_mrpc_prediction_distance", {"prediction": y_pred[i]}, 
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                           "sentence-transformers-sentence-1-distance_threshold": [i, i + 1],
                           "sentence-transformers-sentence-2-distance_threshold": [i, i + 1],
                       }
                       )

description = "INSERT TEXT HERE ABOUT sentence_transformers_mrpc_prediction_distance"
embedding = get_embeddings(description)
vault.create_description("sentence_transformers_mrpc_prediction_distance", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("sentence_transformers_mrpc_prediction_distance", cat, embedding, prop)





In [6]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"])

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))

{'accuracy': 0.678921568627451, 'f1': 0.7456310679611651}
                precision    recall  f1-score   support

not_paraphrase       0.49      0.66      0.56       129
    paraphrase       0.81      0.69      0.75       279

      accuracy                           0.68       408
     macro avg       0.65      0.67      0.66       408
  weighted avg       0.71      0.68      0.69       408



In [7]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "distance:", float(distances_np[i]))

sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 1 distance: 0.35150375962257385
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 0 distance: 1.031471848487854
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 pred: 1 distance: 0.4362468421459198
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO announced Wednesday that it

In [8]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "distance:", float(distances_np[i]))

num_errors: 131
idx: 2
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 pred: 1 distance: 0.4362468421459198
idx: 5
sentence1: Wal-Mart said it would check all of its million-plus domestic workers to ensure they were legally employed .
sentence2: It has also said it would review all of its domestic employees more than 1 million to ensure they have legal status .
true: 1 pred: 0 distance: 0.8789469003677368
idx: 6
sentence1: While dioxin levels in the environment were up last year , they have dropped by 75 percent since the 1970s , said Caswell .
sentence2: The Institute said dioxin levels in the environment have fallen by as much as 76 percent since the 1970s .
true: 0 pred: 1 distance: 0.41990888118743896
idx: 7
sentence1: This integrates with Rational 

In [9]:
vault.create_record_list("st_all_minilm_l6_l2_distance_threshold_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("st_all_minilm_l6_l2_distance_threshold_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "sentence_transformers_mrpc_prediction_distance": [0, len(ds)]
                    })

summary

description = "INSERT TEXT HERE ABOUT st_all_minilm_l6_l2_distance_threshold_mrpc_summary"
embedding = get_embeddings(description)
vault.create_description("st_all_minilm_l6_l2_distance_threshold_mrpc_summary", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("st_all_minilm_l6_l2_distance_threshold_mrpc_summary", cat, embedding, prop)



{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'sentence-transformers/all-MiniLM-L6-v2',
 'device': 'mps',
 'metric': 'l2_distance',
 'distance_threshold': 0.63,
 'num_examples': 408,
 'accuracy': 0.678921568627451,
 'f1': 0.7456310679611651}

In [ ]:
description = "INSERT TEXT HERE ABOUT st_all_minilm_l6_l2_distance_threshold_mrpc process" # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("st_all_minilm_l6_l2_distance_threshold_mrpc", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("st_all_minilm_l6_l2_distance_threshold_mrpc", cat, embedding, prop)